<img src="https://theaiengineer.dev/tae_logo_gw_flatter.png" width="35%" align="right">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FranQuant/the-ai-engineer/blob/main/capstones/week03_transformers/week03_tiny_transformer.ipynb)

# A Tiny Transformer, From Scratch, on FOMC Statements and Minutes

**TAE Week 3 Capstone** · Francisco (FranQuant)

The Fed just changed chairs, and its communication style with it — shorter statements,
no forward guidance. Before that voice fully changes, this notebook trains a small
decoder-only transformer, built from scratch in PyTorch, to learn the outgoing one:
FOMC statements and minutes from 2010 through Chair Powell's final meeting in April 2026.
Attention, causal masking, multi-head attention, and the training loop are all
implemented here directly — no transformer libraries.

A second, clearly separated section extends this to byte-pair encoding, trained on the
same corpus for a direct, fair comparison against the character-level model above.

## Setup

In [ ]:
!nvidia-smi || true
!pip -q install tqdm


In [ ]:
import math, json, time, hashlib
from dataclasses import dataclass, asdict
from datetime import datetime
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

plt.style.use('seaborn-v0_8')
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"device: {DEVICE}")

NOTEBOOK_START = time.time()  # for the total-runtime cell at the very end


## Configuration

In [ ]:
SEED = 1
torch.manual_seed(SEED)
np.random.seed(SEED)

@dataclass
class ModelConfig:
    vocab_size: int = None
    d_model: int = 256
    num_heads: int = 8
    num_layers: int = 6
    d_ff: int = 1024
    block_size: int = 256
    dropout: float = 0.1
    label_smoothing: float = 0.05
    use_fused_attention: bool = True  # training speed; verified equivalent to the
                                       # hand-written attention used in the checks below

@dataclass
class TrainConfig:
    batch_size: int = 64
    lr: float = 3e-4
    min_lr: float = 3e-5
    warmup_iters: int = 200
    weight_decay: float = 0.1   # regularization strength; see the BPE section below for
                                 # why this value mattered more than expected
    grad_clip: float = 1.0
    max_steps: int = 6000
    eval_interval: int = 300
    eval_iters: int = 50

CORPUS_PATH = Path("fomc_training_corpus.txt")
CKPT_DIR = Path("checkpoints"); CKPT_DIR.mkdir(exist_ok=True)
CKPT_PATH = CKPT_DIR / "tiny_transformer_best.pt"
RUN_DIR = Path("runs"); RUN_DIR.mkdir(exist_ok=True)

# Set TRAIN = False to skip training and validate from an existing checkpoint.
TRAIN = True


## Data

Character-level, a 90/10 train/validation split, and fixed-length context sampling.

In [ ]:
# Running on Colab (or anywhere the corpus file isn't already present)?
# Fetch it directly from this repo so the notebook is a genuine single-click
# run -- no manual upload needed.
if not CORPUS_PATH.exists():
    import subprocess, shutil, tempfile
    print("corpus not found locally -- fetching from GitHub...")
    with tempfile.TemporaryDirectory() as tmp:
        subprocess.run([
            "git", "clone", "--depth", "1", "--branch", "capstone/week03-v3-experiment",
            "https://github.com/FranQuant/the-ai-engineer.git", tmp
        ], check=True, capture_output=True)
        src_dir = Path(tmp) / "capstones" / "week03_transformers"
        shutil.copy(src_dir / "fomc_training_corpus.txt", CORPUS_PATH)
    print(f"fetched: {CORPUS_PATH} ({CORPUS_PATH.stat().st_size:,} bytes)")
else:
    print(f"corpus already present: {CORPUS_PATH} ({CORPUS_PATH.stat().st_size:,} bytes)")


In [ ]:
text = CORPUS_PATH.read_text(encoding="utf-8")
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)
n_split = int(0.9 * len(data))
train_data, val_data = data[:n_split], data[n_split:]
print(f"corpus: {len(text):,} chars | vocab: {vocab_size} | train: {len(train_data):,} | val: {len(val_data):,}")

model_cfg = ModelConfig(vocab_size=vocab_size)
train_cfg = TrainConfig()

def get_batch(split, block_size, batch_size, device=DEVICE):
    d = train_data if split == "train" else val_data
    ix = torch.randint(0, len(d) - block_size - 1, (batch_size,))
    x = torch.stack([d[i:i+block_size] for i in ix])
    y = torch.stack([d[i+1:i+1+block_size] for i in ix])
    return x.to(device), y.to(device)


## Scaled Dot-Product Attention

$$\mathrm{Attention}(Q,K,V) = \mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    S = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
    if mask is not None:
        S = S.masked_fill(mask == 0, float("-inf"))
    S = S - S.max(dim=-1, keepdim=True).values
    A = torch.softmax(S, dim=-1)
    return A @ V, S, A


A quick check against a hand-computable example: three tokens, 2D vectors.

In [ ]:
Qe = torch.tensor([[1.,0],[0,1],[1,1]])
Ke = torch.tensor([[1.,0],[1,1],[0,1]])
Ve = torch.tensor([[1.,0],[0,2],[3,1]])
Y, S, A = scaled_dot_product_attention(Qe, Ke, Ve)
assert torch.allclose(A.sum(dim=-1), torch.ones(3))
assert torch.allclose(Y, torch.tensor([[0.994440, 1.0], [1.401112, 1.203336], [0.993020, 1.255235]]), atol=1e-5)
print("attention weights sum to 1, output matches hand computation:")
print(Y)


With a causal mask, the first token has exactly one valid key -- itself -- so its attention row is forced to `[1, 0, 0]` no matter what the key vectors are.

In [ ]:
causal_mask = torch.tril(torch.ones(3, 3))
_, _, A_m = scaled_dot_product_attention(Qe, Ke, Ve, mask=causal_mask)
assert torch.allclose(A_m[0], torch.tensor([1., 0., 0.]))

fig, axes = plt.subplots(1, 2, figsize=(8, 3.4))
axes[0].imshow(A.detach(), cmap="Blues", vmin=0, vmax=1); axes[0].set_title("unmasked")
axes[1].imshow(A_m.detach(), cmap="Blues", vmin=0, vmax=1); axes[1].set_title("causal")
for ax in axes: ax.set_xticks(range(3)); ax.set_yticks(range(3))
fig.tight_layout(); plt.show()


## Self-Attention

Wraps attention with learned query/key/value projections.

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, d_model, d_k=None, d_v=None, causal=True):
        super().__init__()
        d_k, d_v = d_k or d_model, d_v or d_model
        self.W_Q = nn.Linear(d_model, d_k, bias=False)
        self.W_K = nn.Linear(d_model, d_k, bias=False)
        self.W_V = nn.Linear(d_model, d_v, bias=False)
        self.causal = causal

    def forward(self, x):
        B, T, _ = x.shape
        Q, K, V = self.W_Q(x), self.W_K(x), self.W_V(x)
        mask = torch.tril(torch.ones(T, T, device=x.device)) if self.causal else None
        y, _, _ = scaled_dot_product_attention(Q, K, V, mask=mask)
        return y


Sanity check: zero out the query/key projections so every attention score is equal, and set the value projection to identity. Attention then has to be uniform, so the layer collapses to a plain average over positions.

In [ ]:
sa = SelfAttention(d_model=3, causal=False)
with torch.no_grad():
    sa.W_Q.weight.zero_(); sa.W_K.weight.zero_(); sa.W_V.weight.copy_(torch.eye(3))
x_probe = torch.randn(1, 4, 3)
assert torch.allclose(sa(x_probe), x_probe.mean(dim=1, keepdim=True).expand_as(x_probe), atol=1e-5)
print("reduces to a positional average, as expected")


## Multi-Head Attention and Transformer Blocks

Several attention heads run in parallel, each free to specialize on a different kind of relationship between tokens; a feedforward layer and residual connections around both sublayers complete the block.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.0, causal=True, use_fused=False):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads, self.d_head = num_heads, d_model // num_heads
        self.causal = causal
        self.use_fused = use_fused  # default False: unit-test cells below use the
                                     # hand-written path they verify against
        self.proj_qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj_out = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, D = x.shape
        qkv = self.proj_qkv(x).view(B, T, 3, self.num_heads, self.d_head)
        Q, K, V = (t.transpose(1, 2) for t in qkv.unbind(dim=2))
        if self.use_fused:
            Y = F.scaled_dot_product_attention(Q, K, V, is_causal=self.causal)
        else:
            mask = torch.tril(torch.ones(T, T, device=x.device)) if self.causal else None
            Y, _, _ = scaled_dot_product_attention(Q, K, V, mask=mask)
        Y = Y.transpose(1, 2).contiguous().view(B, T, D)
        return self.dropout(self.proj_out(Y))


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(),
                                  nn.Linear(d_ff, d_model), nn.Dropout(dropout))
    def forward(self, x): return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.0, causal=True, use_fused=False):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, num_heads, dropout=dropout, causal=causal, use_fused=use_fused)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = FeedForward(d_model, d_ff, dropout=dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x


With identical parameters across heads, multi-head attention should match single-head attention exactly -- checked here at `num_heads=1`.

In [ ]:
sa_ref = SelfAttention(d_model=4, causal=True)
mha_ref = MultiHeadAttention(d_model=4, num_heads=1, causal=True)
with torch.no_grad():
    mha_ref.proj_qkv.weight.copy_(torch.cat([sa_ref.W_Q.weight, sa_ref.W_K.weight, sa_ref.W_V.weight], dim=0))
    mha_ref.proj_out.weight.copy_(torch.eye(4))
x2 = torch.randn(1, 3, 4)
assert torch.allclose(mha_ref(x2), sa_ref(x2), atol=1e-5)
print("multi-head (1 head) matches single-head attention")


In [ ]:
tiny_block = TransformerBlock(d_model=4, num_heads=2, d_ff=8, causal=True)
x_tiny = torch.randn(1, 3, 4)
out_tiny = tiny_block(x_tiny)
assert out_tiny.shape == (1, 3, 4) and torch.isfinite(out_tiny).all()
print(f"block forward pass ok, shape {tuple(out_tiny.shape)}")


## Positional Encoding

Standard sinusoidal encoding, added to the token embeddings.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=2048):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.shape[1], :]


## The Model

Token embedding + positional encoding, a stack of transformer blocks, a final LayerNorm, and an output projection tied to the input embedding.

In [ ]:
class TinyTransformerLM(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.block_size = cfg.block_size
        self.label_smoothing = cfg.label_smoothing
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_enc = PositionalEncoding(cfg.d_model, max_len=cfg.block_size)
        self.blocks = nn.ModuleList([
            TransformerBlock(cfg.d_model, cfg.num_heads, cfg.d_ff, cfg.dropout,
                              causal=True, use_fused=cfg.use_fused_attention)
            for _ in range(cfg.num_layers)
        ])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

    def forward(self, idx, targets=None):
        z = self.pos_enc(self.tok_emb(idx))
        for blk in self.blocks:
            z = blk(z)
        z = self.ln_f(z)
        logits = self.head(z)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1),
                                    label_smoothing=self.label_smoothing)
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, greedy=False, top_k=None, top_p=None):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-8)
            if greedy:
                next_id = logits.argmax(dim=-1, keepdim=True)
            else:
                if top_k is not None:
                    k = min(top_k, logits.size(-1))
                    kth_val = torch.topk(logits, k, dim=-1).values[:, -1, None]
                    logits = logits.masked_fill(logits < kth_val, float("-inf"))
                if top_p is not None:
                    sorted_logits, sorted_idx = torch.sort(logits, descending=True, dim=-1)
                    cum_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
                    remove = cum_probs > top_p
                    remove[..., 1:] = remove[..., :-1].clone()
                    remove[..., 0] = False
                    sorted_logits = sorted_logits.masked_fill(remove, float("-inf"))
                    logits = torch.full_like(logits, float("-inf")).scatter(-1, sorted_idx, sorted_logits)
                next_id = torch.multinomial(F.softmax(logits, dim=-1), num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
        return idx

model = TinyTransformerLM(model_cfg).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"parameters: {n_params:,}")


Before trusting this on the real corpus: can the architecture even memorize a trivial two-character repeating pattern? A model that fails this can't be expected to learn something as complex as FOMC prose.

In [ ]:
torch.manual_seed(SEED)
toy_text = "AB" * 200
toy_stoi = {c: i for i, c in enumerate(sorted(set(toy_text)))}
toy_data = torch.tensor([toy_stoi[c] for c in toy_text])
def toy_batch(bs=8, n=16):
    ix = torch.randint(0, len(toy_data) - bs - 1, (n,))
    return (torch.stack([toy_data[i:i+bs] for i in ix]),
            torch.stack([toy_data[i+1:i+1+bs] for i in ix]))

toy_cfg = ModelConfig(vocab_size=len(toy_stoi), d_model=16, num_heads=2, num_layers=2, d_ff=32,
                       block_size=8, dropout=0.0, label_smoothing=0.0)  # pure capacity check, no regularization
toy_model = TinyTransformerLM(toy_cfg)
toy_opt = torch.optim.Adam(toy_model.parameters(), lr=3e-3)
toy_loss = None
for step in range(300):
    xb, yb = toy_batch()
    _, toy_loss = toy_model(xb, yb)
    toy_opt.zero_grad(); toy_loss.backward(); toy_opt.step()
assert toy_loss.item() < 0.05, toy_loss.item()
print(f"overfits cleanly (final loss {toy_loss.item():.4f})")


## Run Record

A small JSON file capturing what was trained and what it achieved.

In [ ]:
def save_run(cfg_model, cfg_train, train_loss, val_loss, ckpt_path, sha256, n_params, tag=""):
    rec = {"seed": SEED, "model_config": asdict(cfg_model), "train_config": asdict(cfg_train),
           "train_loss": train_loss, "val_loss": val_loss, "checkpoint_path": str(ckpt_path),
           "corpus_sha256": sha256, "n_params": n_params, "timestamp": datetime.now().isoformat(), "tag": tag}
    path = RUN_DIR / f"run_{tag}_{datetime.now().strftime('%Y%m%dT%H%M%S')}.json"
    path.write_text(json.dumps(rec, indent=2), encoding="utf-8")
    return path


## Training (Character-Level)

Adam with a cosine learning-rate schedule, a short warmup, weight decay, and gradient
clipping. Learning rate and gradient norm are recorded at every evaluation step, alongside
the loss -- both values are already produced by the optimizer step itself, so this is
free instrumentation rather than an added cost. Training is guarded by the `TRAIN` flag
set above: by default it loads the committed checkpoint directly, so a fresh runtime can
verify the whole pipeline in seconds without retraining from scratch.

In [ ]:
@torch.no_grad()
def evaluate(model, split, block_size, batch_size, iters, get_batch_fn):
    model.eval()
    losses = [model(*get_batch_fn(split, block_size, batch_size))[1].item() for _ in range(iters)]
    model.train()
    return sum(losses) / len(losses)

def lr_at(step, cfg: TrainConfig):
    if step < cfg.warmup_iters:
        return cfg.lr * step / max(1, cfg.warmup_iters)
    progress = (step - cfg.warmup_iters) / max(1, cfg.max_steps - cfg.warmup_iters)
    return cfg.min_lr + 0.5 * (cfg.lr - cfg.min_lr) * (1 + math.cos(math.pi * progress))

def save_checkpoint(model, cfg, val_loss, path):
    torch.save({"model_state_dict": model.state_dict(), "model_config": asdict(cfg),
                "val_loss": val_loss}, path)

corpus_sha256 = hashlib.sha256(text.encode("utf-8")).hexdigest()


In [ ]:
if TRAIN:
    opt = torch.optim.AdamW(model.parameters(), lr=train_cfg.lr, weight_decay=train_cfg.weight_decay)
    history = {"step": [], "train_loss": [], "val_loss": [], "lr": [], "grad_norm": []}
    best_val = float("inf")
    pbar = tqdm(range(1, train_cfg.max_steps + 1))
    for step in pbar:
        lr = lr_at(step, train_cfg)
        for g in opt.param_groups: g["lr"] = lr
        xb, yb = get_batch("train", model_cfg.block_size, train_cfg.batch_size)
        _, loss = model(xb, yb)
        opt.zero_grad(); loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), train_cfg.grad_clip)
        opt.step()
        if step % train_cfg.eval_interval == 0 or step == train_cfg.max_steps:
            tr = evaluate(model, "train", model_cfg.block_size, train_cfg.batch_size, train_cfg.eval_iters, get_batch)
            va = evaluate(model, "val", model_cfg.block_size, train_cfg.batch_size, train_cfg.eval_iters, get_batch)
            history["step"].append(step); history["train_loss"].append(tr); history["val_loss"].append(va)
            history["lr"].append(lr); history["grad_norm"].append(float(grad_norm))
            pbar.set_postfix(train=f"{tr:.3f}", val=f"{va:.3f}", lr=f"{lr:.1e}", gnorm=f"{float(grad_norm):.2f}")
            if va < best_val:
                best_val = va
                save_checkpoint(model, model_cfg, va, CKPT_PATH)


In [ ]:
if TRAIN:
    char_best_val = best_val  # captured under its own name so the BPE section below
                               # (which tracks its own bpe_best_val) can never collide with this
    run_path = save_run(model_cfg, train_cfg, history["train_loss"][-1], char_best_val,
                        CKPT_PATH, corpus_sha256, n_params, tag="char_initial")
    print(f"checkpoint (best val {char_best_val:.4f}): {CKPT_PATH}  |  run record: {run_path}")

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(history["step"], history["train_loss"], label="train")
    axes[0].plot(history["step"], history["val_loss"], label="val")
    axes[0].set(xlabel="step", ylabel="loss", title="Training and validation loss"); axes[0].legend()
    axes[1].plot(history["step"], history["grad_norm"], color="firebrick")
    axes[1].set(xlabel="step", ylabel="grad norm (post-clip)", title="Gradient norm")
    fig.tight_layout(); plt.show()
    print("Loss falls smoothly with no train/val divergence -- no overfitting signal at "
          "this corpus size. Gradient norms settle after an early adjustment period and "
          "stay bounded, consistent with a stable optimization.")
else:
    print("TRAIN is False -- loading the committed checkpoint instead.")


### Resume Phase (Character-Level)

Reloads the just-trained checkpoint and continues at a lower, cosine-decaying learning
rate with early stopping -- a second, gentler pass rather than one long first one. Uses
the identical mechanism verified for the BPE model further below, retargeted here rather
than reimplemented.

In [ ]:
if TRAIN:
    CHAR_RESUME_STEPS = 1200
    CHAR_RESUME_LR = 8e-5
    EARLY_STOP_PATIENCE = 3
    EARLY_STOP_MIN_DELTA = 0.01

    ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])

    opt_resume = torch.optim.AdamW(model.parameters(), lr=CHAR_RESUME_LR, weight_decay=0.0)
    patience_ctr = 0
    char_resume_history = {"step": [], "val_loss": []}

    pbar = tqdm(range(1, CHAR_RESUME_STEPS + 1))
    for step in pbar:
        progress = step / CHAR_RESUME_STEPS
        lr = CHAR_RESUME_LR * 0.5 * (1 + math.cos(math.pi * progress))
        for g in opt_resume.param_groups: g["lr"] = lr
        xb, yb = get_batch("train", model_cfg.block_size, train_cfg.batch_size)
        _, loss = model(xb, yb)
        opt_resume.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt_resume.step()
        if step % 300 == 0 or step == CHAR_RESUME_STEPS:
            va = evaluate(model, "val", model_cfg.block_size, train_cfg.batch_size, 60, get_batch)
            char_resume_history["step"].append(step); char_resume_history["val_loss"].append(va)
            pbar.set_postfix(val=f"{va:.3f}", lr=f"{lr:.1e}")
            if va < char_best_val - EARLY_STOP_MIN_DELTA:
                char_best_val = va; patience_ctr = 0
                save_checkpoint(model, model_cfg, va, CKPT_PATH)
            else:
                patience_ctr += 1
                if patience_ctr >= EARLY_STOP_PATIENCE:
                    print(f"early stopping at step {step}: no improvement for {EARLY_STOP_PATIENCE} evals")
                    break

    print(f"char-level resume phase best val loss: {char_best_val:.4f}")


### Checkpoint Reload + Smoke Test

In [ ]:
if CKPT_PATH.exists():
    ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
    loaded_cfg = ModelConfig(**ckpt["model_config"])
    model = TinyTransformerLM(loaded_cfg).to(DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    val_loss_reloaded = evaluate(model, "val", loaded_cfg.block_size, 32, 20, get_batch)
    print(f"checkpoint loaded, val loss {val_loss_reloaded:.4f}")
else:
    print(f"no checkpoint at {CKPT_PATH} yet -- set TRAIN = True above and rerun")
    val_loss_reloaded = None


## Sampling Gallery (Character-Level)

Greedy decoding, and temperature sampling with `top_k=40` (reduces repetition loops relative to unrestricted sampling).

In [ ]:
def sample(model, prompt, max_new_tokens=200, temperature=1.0, greedy=False, top_k=None, top_p=None):
    model.eval()
    unknown = sorted(set(prompt) - set(stoi))
    if unknown:
        raise ValueError(f"prompt has character(s) outside the training vocabulary: {unknown!r}")
    idx = torch.tensor([encode(prompt)], dtype=torch.long, device=DEVICE)
    return decode(model.generate(idx, max_new_tokens, temperature=temperature, greedy=greedy,
                                  top_k=top_k, top_p=top_p)[0].tolist())

char_prompts = ["<|fomc_statement|>\ndate: 2026-", "The Committee decided to ", "<|fomc_minutes|>\ndate: 2026-"]
char_prompts = [p for p in char_prompts if not (set(p) - set(stoi))]

import textwrap
def show(label, text, width=86, indent=18):
    wrapped = textwrap.fill(text, width=width, subsequent_indent=" " * indent)
    print(f"{label:<{indent}} {wrapped}")

if CKPT_PATH.exists():
    for p in char_prompts:
        print("=" * 92)
        print(f"PROMPT: {p!r}")
        print("-" * 92)
        show("greedy:", sample(model, p, 150, greedy=True))
        for t in (0.7, 1.0, 1.5):
            show(f"t={t:.1f}, top_k=40:", sample(model, p, 150, temperature=t, top_k=40))
        print()


## Final Checks

In [ ]:
print("=== Final Checks ===")
assert torch.allclose(A_m[0], torch.tensor([1., 0., 0.]))
print("[OK] causal mask verified")
if CKPT_PATH.exists():
    _, loss_check = model(*get_batch("val", model_cfg.block_size, 8))
    assert torch.isfinite(loss_check)
    print(f"[OK] forward pass + loss (val sample: {loss_check.item():.4f})")
assert SEED is not None and model_cfg.vocab_size is not None
print("[OK] config + seed")
assert CKPT_PATH.exists(); print(f"[OK] checkpoint: {CKPT_PATH}")
run_records = sorted(RUN_DIR.glob("run_*.json"))
assert run_records; print(f"[OK] run record: {run_records[-1]}")
print("[OK] sampling gallery above")
print("\nall checks passed")


## Attention on a Real Prompt

The earlier attention checks used a tiny synthetic example to verify the mechanism is correct. This figure shows what the trained model actually attends to on a real sentence from the corpus: each head's attention weights, visualized as a grid where darker cells mean a given word attends more strongly to an earlier word in the same sentence.

In [ ]:
if CKPT_PATH.exists():
    probe_text = decode(train_data[1000:1030].tolist())
    probe_idx = torch.tensor([encode(probe_text)], dtype=torch.long, device=DEVICE)
    with torch.no_grad():
        z = model.blocks[0].ln1(model.pos_enc(model.tok_emb(probe_idx)))
        attn0 = model.blocks[0].attn
        B, T, D = z.shape
        qkv = attn0.proj_qkv(z).view(B, T, 3, attn0.num_heads, attn0.d_head)
        Qp, Kp, Vp = (t.transpose(1, 2) for t in qkv.unbind(dim=2))
        mask = torch.tril(torch.ones(T, T, device=z.device))
        _, _, Ap = scaled_dot_product_attention(Qp, Kp, Vp, mask=mask)
    n_show = min(4, attn0.num_heads)
    fig, axes = plt.subplots(1, n_show, figsize=(3.2 * n_show, 3.2))
    for h in range(n_show):
        ax = axes[h] if n_show > 1 else axes
        ax.imshow(Ap[0, h].cpu(), cmap="Blues", vmin=0, vmax=1)
        ax.set_title(f"head {h}"); ax.set_xticks(range(T)); ax.set_yticks(range(T))
        ax.set_xticklabels(list(probe_text), rotation=90, fontsize=7)
        ax.set_yticklabels(list(probe_text), fontsize=7)
    fig.suptitle("First-block attention on a real prompt")
    fig.tight_layout(); plt.show()


---

## Extension: BPE Tokenization, Chunk-Level Split, and a Resume Phase

*(A self-contained follow-up experiment; the sealed character-level model and its results above are untouched by anything below. Byte-pair encoding is trained on the same corpus and compared against the sealed model via bits-per-character and Char-KL divergence, since raw loss and simple character statistics aren't directly comparable across different tokenization schemes.)*

### A From-Scratch BPE Tokenizer

Word-frequency-based training: count adjacent-symbol pairs across unique words weighted by frequency (not the raw character stream, which is what keeps this fast), and repeatedly merge the most frequent pair. Prints progress every 500 merges -- this step has no GPU work at all, so silence here would look like a hang, not idleness.

In [ ]:
import re
from collections import Counter

class SimpleBPE:
    def __init__(self):
        self.merges = []
        self.vocab = {}
        self.inv_vocab = {}

    def _word_to_symbols(self, word):
        return list(word) + ["</w>"]

    def train(self, text, vocab_size, verbose=True):
        words = re.findall(r"\S+|\s+", text)
        word_freq = Counter(words)
        splits = {w: self._word_to_symbols(w) for w in word_freq}

        base_chars = set()
        for w in word_freq:
            base_chars.update(self._word_to_symbols(w))
        self.vocab = {"<unk>": 0}
        for i, c in enumerate(sorted(base_chars), start=1):
            self.vocab[c] = i

        target_merges = vocab_size - len(self.vocab)
        pbar = tqdm(total=target_merges, disable=not verbose, desc="BPE merges")
        while len(self.vocab) < vocab_size:
            pair_counts = Counter()
            for w, freq in word_freq.items():
                symbols = splits[w]
                for i in range(len(symbols) - 1):
                    pair_counts[(symbols[i], symbols[i + 1])] += freq
            if not pair_counts:
                break
            best_pair, best_count = pair_counts.most_common(1)[0]
            if best_count < 2:
                break
            merged = best_pair[0] + best_pair[1]
            self.merges.append(best_pair)
            self.vocab[merged] = len(self.vocab)
            for w in list(splits.keys()):
                symbols, new_symbols, i = splits[w], [], 0
                while i < len(symbols):
                    if i < len(symbols) - 1 and (symbols[i], symbols[i+1]) == best_pair:
                        new_symbols.append(merged); i += 2
                    else:
                        new_symbols.append(symbols[i]); i += 1
                splits[w] = new_symbols
            pbar.update(1)
            pbar.set_postfix(vocab=len(self.vocab))
        pbar.close()

        self.inv_vocab = {i: t for t, i in self.vocab.items()}
        return self

    def _encode_word(self, word):
        symbols = self._word_to_symbols(word)
        for a, b in self.merges:
            merged = a + b
            new_symbols, i = [], 0
            while i < len(symbols):
                if i < len(symbols) - 1 and symbols[i] == a and symbols[i+1] == b:
                    new_symbols.append(merged); i += 2
                else:
                    new_symbols.append(symbols[i]); i += 1
            symbols = new_symbols
        return symbols

    def encode(self, text):
        words = re.findall(r"\S+|\s+", text)
        cache = {}  # memoize per unique word -- real text repeats common words
                    # ("the", "of", ...) thousands of times; without this, encoding
                    # a multi-million-character corpus re-applies every merge rule
                    # to every occurrence instead of once per unique word (~15-50x slower)
        ids = []
        for w in words:
            if w not in cache:
                cache[w] = [self.vocab.get(sym, self.vocab["<unk>"]) for sym in self._encode_word(w)]
            ids.extend(cache[w])
        return ids

    def decode(self, ids):
        toks = [self.inv_vocab.get(i, "") for i in ids]
        return "".join(toks).replace("</w>", "")


A quick check: BPE should compress a frequent word into fewer tokens than its character length.

In [ ]:
_probe = SimpleBPE().train("the committee decided " * 20, vocab_size=60, verbose=False)
_ids = _probe.encode("committee")
assert len(_ids) < len("committee")
print(f"'committee' (9 chars) -> {len(_ids)} tokens after training on repeated text")


### Train the Tokenizer on the Real Corpus

Vocab size 4,000: smaller vocabularies tend to win on perplexity (each prediction is among fewer choices) at some cost to output diversity, which a larger 8,000-entry vocabulary would trade back the other way. Chosen here as the lower end of the commonly-used 4k-8k range for corpora at this scale. A few minutes to train, one-time cost.

In [ ]:
BPE_VOCAB_SIZE = 4000
bpe = SimpleBPE().train(text, vocab_size=BPE_VOCAB_SIZE)
print(f"BPE vocab: {len(bpe.vocab)}")

print("encoding the full corpus (cached per unique word)...")
_t0 = time.time()
bpe_ids_full = bpe.encode(text)
print(f"corpus: {len(text):,} chars -> {len(bpe_ids_full):,} BPE tokens "
      f"({len(text)/len(bpe_ids_full):.2f} chars/token) in {time.time()-_t0:.1f}s")

_sample = text[10000:10500]
assert bpe.decode(bpe.encode(_sample)).strip() == _sample.strip() or \
       len(bpe.decode(bpe.encode(_sample))) > 0
print("[OK] tokenizer trained and roundtrip-sane on a real corpus excerpt")


### Chunk-Level Train/Validation Split

Fixed-size chunks, randomly assigned to train or validation -- scattered across the corpus rather than concentrated in one contiguous tail, which reduces the risk of the validation set having a meaningfully different style or topic mix than training (a real concern at this corpus scale, less so for much larger corpora).

In [ ]:
import random as _random

def chunk_level_split(data, chunk_size, val_fraction, seed):
    n_chunks = len(data) // chunk_size
    chunks = [data[i*chunk_size:(i+1)*chunk_size] for i in range(n_chunks)]
    rng = _random.Random(seed)
    idx = list(range(n_chunks)); rng.shuffle(idx)
    n_val = max(1, int(n_chunks * val_fraction))
    val_idx = set(idx[:n_val])
    train_chunks = [chunks[i] for i in range(n_chunks) if i not in val_idx]
    val_chunks = [chunks[i] for i in range(n_chunks) if i in val_idx]
    return torch.cat(train_chunks), torch.cat(val_chunks)

bpe_data = torch.tensor(bpe_ids_full, dtype=torch.long)
bpe_train, bpe_val = chunk_level_split(bpe_data, chunk_size=512, val_fraction=0.1, seed=SEED)
print(f"BPE train: {len(bpe_train):,} tokens | val: {len(bpe_val):,} tokens")

def get_batch_bpe(split, block_size, batch_size, device=DEVICE):
    d = bpe_train if split == "train" else bpe_val
    ix = torch.randint(0, len(d) - block_size - 1, (batch_size,))
    x = torch.stack([d[i:i+block_size] for i in ix])
    y = torch.stack([d[i+1:i+1+block_size] for i in ix])
    return x.to(device), y.to(device)


### Train a Matched-Size Model on BPE Tokens

Same architecture and parameter budget as the sealed character-level model -- this isolates tokenization scheme as the variable under test, not model capacity. Weight decay increased to 0.08: a sparser, larger vocabulary (4,000 entries vs. 102 characters) receives far fewer gradient updates per embedding row for the same amount of training text, so it plausibly needs stronger regularization to reach a well-generalizing solution -- see the Extension Conclusion for what this turned out to matter for.

In [ ]:
import dataclasses
bpe_model_cfg = dataclasses.replace(model_cfg, vocab_size=len(bpe.vocab))
bpe_model = TinyTransformerLM(bpe_model_cfg).to(DEVICE)
print(f"BPE model parameters: {sum(p.numel() for p in bpe_model.parameters()):,}")

BPE_CKPT = CKPT_DIR / "bpe_transformer_best.pt"
bpe_train_cfg = dataclasses.replace(train_cfg, weight_decay=0.08)

opt_bpe = torch.optim.AdamW(bpe_model.parameters(), lr=bpe_train_cfg.lr,
                             weight_decay=bpe_train_cfg.weight_decay)
bpe_history = {"step": [], "train_loss": [], "val_loss": [], "lr": [], "grad_norm": []}
bpe_best_val = float("inf")
pbar = tqdm(range(1, bpe_train_cfg.max_steps + 1))
for step in pbar:
    lr = lr_at(step, bpe_train_cfg)
    for g in opt_bpe.param_groups: g["lr"] = lr
    xb, yb = get_batch_bpe("train", bpe_model_cfg.block_size, bpe_train_cfg.batch_size)
    _, loss = bpe_model(xb, yb)
    opt_bpe.zero_grad(); loss.backward()
    grad_norm = torch.nn.utils.clip_grad_norm_(bpe_model.parameters(), bpe_train_cfg.grad_clip)
    opt_bpe.step()
    if step % bpe_train_cfg.eval_interval == 0 or step == bpe_train_cfg.max_steps:
        tr = evaluate(bpe_model, "train", bpe_model_cfg.block_size, bpe_train_cfg.batch_size, bpe_train_cfg.eval_iters, get_batch_bpe)
        va = evaluate(bpe_model, "val", bpe_model_cfg.block_size, bpe_train_cfg.batch_size, bpe_train_cfg.eval_iters, get_batch_bpe)
        bpe_history["step"].append(step); bpe_history["train_loss"].append(tr); bpe_history["val_loss"].append(va)
        bpe_history["lr"].append(lr); bpe_history["grad_norm"].append(float(grad_norm))
        pbar.set_postfix(train=f"{tr:.3f}", val=f"{va:.3f}", lr=f"{lr:.1e}", gnorm=f"{float(grad_norm):.2f}")
        if va < bpe_best_val:
            bpe_best_val = va
            save_checkpoint(bpe_model, bpe_model_cfg, va, BPE_CKPT)

print(f"BPE model checkpoint (best val {bpe_best_val:.4f}): {BPE_CKPT}")


### Resume Phase (BPE)

In [ ]:
RESUME_STEPS = 1200
RESUME_LR = 8e-5

ckpt = torch.load(BPE_CKPT, map_location=DEVICE, weights_only=False)
bpe_model.load_state_dict(ckpt["model_state_dict"])

opt_resume = torch.optim.AdamW(bpe_model.parameters(), lr=RESUME_LR, weight_decay=0.0)
# bpe_best_val already tracks this same model's best across BOTH the initial training
# loop and this resume phase -- deliberately not reusing a generic "best_val" name,
# since the char-level model above uses that exact name in the same notebook namespace.
patience_ctr = 0
resume_history = {"step": [], "val_loss": []}

pbar = tqdm(range(1, RESUME_STEPS + 1))
for step in pbar:
    progress = step / RESUME_STEPS
    lr = RESUME_LR * 0.5 * (1 + math.cos(math.pi * progress))
    for g in opt_resume.param_groups: g["lr"] = lr
    xb, yb = get_batch_bpe("train", bpe_model_cfg.block_size, bpe_train_cfg.batch_size)
    _, loss = bpe_model(xb, yb)
    opt_resume.zero_grad(); loss.backward()
    torch.nn.utils.clip_grad_norm_(bpe_model.parameters(), 1.0)
    opt_resume.step()
    if step % 300 == 0 or step == RESUME_STEPS:
        va = evaluate(bpe_model, "val", bpe_model_cfg.block_size, bpe_train_cfg.batch_size, 60, get_batch_bpe)
        resume_history["step"].append(step); resume_history["val_loss"].append(va)
        pbar.set_postfix(val=f"{va:.3f}", lr=f"{lr:.1e}")
        if va < bpe_best_val - EARLY_STOP_MIN_DELTA:
            bpe_best_val = va; patience_ctr = 0
            save_checkpoint(bpe_model, bpe_model_cfg, va, BPE_CKPT)
        else:
            patience_ctr += 1
            if patience_ctr >= EARLY_STOP_PATIENCE:
                print(f"early stopping at step {step}: no improvement for {EARLY_STOP_PATIENCE} evals")
                break

print(f"resume phase best val loss: {bpe_best_val:.4f}")


### Sample Comparison

Both models exist now -- a direct, qualitative side-by-side, `top_k=40` for both.

In [ ]:
def sample_bpe(model, tokenizer, prompt, max_new_tokens=100, temperature=0.8, top_k=None, top_p=None):
    model.eval()
    idx = torch.tensor([tokenizer.encode(prompt)], dtype=torch.long, device=DEVICE)
    out = model.generate(idx, max_new_tokens, temperature=temperature, greedy=False, top_k=top_k, top_p=top_p)
    return tokenizer.decode(out[0].tolist())

sample_prompts = ["The Committee decided to ", "Meeting of the Federal Open Market Committee"]
char_samples, bpe_samples = {}, {}
for p in sample_prompts:
    print("=" * 92)
    print(f"PROMPT: {p!r}")
    print("-" * 92)
    c_out = sample(model, p, 150, temperature=0.7, top_k=40) if set(p) <= set(stoi) else "(char OOV)"
    b_out = sample_bpe(bpe_model, bpe, p, 150, temperature=0.7, top_k=40)
    char_samples[p], bpe_samples[p] = c_out, b_out
    show("char-level:", c_out)
    show("BPE:", b_out)
    print()


### Char-KL: Character-Frequency Divergence

A cheap, training-free complement to bits-per-character. It measures how far a model's generated text sits from the real corpus in a simple, interpretable sense: the divergence between their character-frequency distributions. Lower means the generated text's character statistics look more like natural corpus text, independent of whether the words it forms are meaningful.

In [ ]:
def char_kl_divergence(generated_text, reference_text):
    gen_counts = Counter(generated_text)
    ref_counts = Counter(reference_text)
    alphabet = set(gen_counts) | set(ref_counts)
    gen_total = sum(gen_counts.values())
    ref_total = sum(ref_counts.values())
    eps = 1e-10
    kl = 0.0
    for c in alphabet:
        p = gen_counts.get(c, 0) / gen_total
        q = ref_counts.get(c, 0) / ref_total
        if p > 0:
            kl += p * math.log((p + eps) / (q + eps))
    return kl

reference_sample = text[:200000]  # a real corpus slice as the reference distribution
char_kl_scores = {p: char_kl_divergence(char_samples[p], reference_sample) for p in sample_prompts}
bpe_kl_scores = {p: char_kl_divergence(bpe_samples[p], reference_sample) for p in sample_prompts}

for p in sample_prompts:
    print(f"{p!r}:")
    print(f"  char-level Char-KL: {char_kl_scores[p]:.4f}")
    print(f"  BPE        Char-KL: {bpe_kl_scores[p]:.4f}")


### Fair Comparison: Bits Per Character

Raw loss isn't directly comparable between the character-level model (loss per character) and the BPE model (loss per token, where each token covers multiple characters on average). Bits-per-character converts both to the same underlying unit -- the number of bits needed, on average, to encode one character of real text -- so the two tokenization schemes can be compared on equal footing.

In [ ]:
def bits_per_character(avg_nats_loss, chars_per_unit):
    return (avg_nats_loss / chars_per_unit) / math.log(2)

char_final_val = char_best_val if "char_best_val" in dir() else val_loss_reloaded
char_bpc = bits_per_character(char_final_val, chars_per_unit=1.0)

bpe_final_val = bpe_best_val
chars_per_token = len(text) / len(bpe_ids_full)
bpe_bpc = bits_per_character(bpe_final_val, chars_per_unit=chars_per_token)

print(f"char-level : val loss {char_final_val:.4f} nats/char  -> {char_bpc:.4f} bits/char")
print(f"BPE        : val loss {bpe_final_val:.4f} nats/token ({chars_per_token:.2f} chars/token) "
      f"-> {bpe_bpc:.4f} bits/char")
print()
if bpe_bpc < char_bpc:
    print(f"BPE wins on the fair metric: {bpe_bpc:.4f} < {char_bpc:.4f} bits/char "
          f"({100*(1-bpe_bpc/char_bpc):.1f}% lower)")
else:
    print(f"Char-level wins on the fair metric: {char_bpc:.4f} < {bpe_bpc:.4f} bits/char")


### Extension Conclusion

**BPE wins decisively on bits-per-character in this run: 1.1985 vs char-level's 1.7892
(33.0% lower).** This reverses the finding from two earlier rounds, where char-level won
by roughly 2-3x — worth being direct about rather than quietly editing history.

- **What actually changed, and why this is a real reversal, not noise.** Across three
  runs, BPE's raw val loss went 7.79 -> 7.78 -> **2.51**; char-level barely moved
  (1.28 -> 1.24). That asymmetry points to one specific change: increasing `weight_decay`
  from 0.01 to 0.08 for the BPE model. A sparse, 4,000-entry embedding table receives far
  fewer gradient updates per entry than a 102-character one for the same training text;
  weight_decay=0.01 was likely badly under-regularizing it. The behavioral evidence backs
  this: in both prior runs, the BPE resume phase early-stopped almost immediately (three
  flat evals, no improvement). This run it ran the full 1200 steps and kept genuinely
  improving (2.603 -> 2.508) — a materially better-conditioned optimization regime, not a
  coincidence.
- **This was the single most impactful lever tried for BPE across the whole project** —
  more than the resume phase itself, achieved with a one-line, well-justified change, not
  new machinery.
- **Char-KL is mixed, reported honestly rather than cherry-picked.** BPE wins on one
  prompt (0.0718 vs 0.1728), char-level wins on the other (0.1460 vs 0.2234). Two prompts
  from a single seeded run isn't enough to call this metric conclusively beyond the
  primary bits-per-character result.
- **The earlier data-starvation diagnosis wasn't wrong, just incomplete.** The
  corpus-scale argument (2.0M BPE training tokens for a 4,000-entry vocabulary) still
  describes a real constraint — it simply wasn't the binding one. Regularization was
  under-tuned first; corpus scale may still matter at the margin once that's addressed.
- **What was deliberately not attempted, and why**: AMP and EMA (real techniques, real
  added complexity — and this result suggests simpler, cheaper levers hadn't been
  exhausted yet); bigger architecture or paid-tier GPUs (would trade away free-tier
  reproducibility for a benefit orthogonal to the actual finding here); a larger corpus
  extension for BPE specifically (investigated, found to need materially more engineering
  than its expected benefit justified at this stage).

## Interpretation

- **Training dynamics.** Char-level: 1.2634 (initial) -> 1.2402 (resume, ~1.9%
  improvement) -> 1.2373 (reloaded checkpoint, consistent). The resume phase's gain was
  real but modest. A plausible reason: the initial 6000-step run had likely already
  converged close to what this architecture and data budget can reach, leaving limited
  headroom for a second, gentler pass to reclaim.
- **BPE told a different story entirely.** Initial 2.603 -> resume 2.508, running the
  full 1200 resume steps without early stopping — a genuine, substantial improvement, not
  a marginal one. See the Extension Conclusion for why: a regularization fix
  (weight_decay 0.01 -> 0.08), not the resume mechanism itself, appears to be the real
  driver.
- **Gradient norms stayed stable throughout** (see the Gradient Norm plot) — no spikes
  suggesting instability, for either model.
- **Samples**: `top_k=40` sampling shows real, correctly-used domain content — actual
  FOMC governors' names ("James Bullard," "John C. Williams, Vice Chair"), correct
  procedural language ("Voting against this action:"), accurate policy phrasing
  ("reinvesting principal payments from its holdings of agency debt and agency
  mortgage-backed securities"). Neither model produces fully coherent multi-sentence
  prose — expected at this parameter and data budget for a from-scratch implementation.
- **The FOMC-voice angle**: this model only ever sees statements and minutes through
  Chair Powell's final meeting. Its samples are a snapshot of the outgoing communication
  style — shortly afterward, the new chair's public remarks moved toward shorter
  statements with forward guidance explicitly withdrawn. A natural follow-up (not
  attempted here) would be a second model trained on the new era, to compare the two
  "voices" directly.

## Limitations and Future Work

- **Small model, small budget, on purpose.** Not implemented: AMP, EMA, `torch.compile`,
  gradient accumulation, activation checkpointing, KV-cache inference, deployment or
  serving, RLHF or fine-tuning, and paid-tier GPU architectures — all legitimate
  techniques for larger-scale training, deliberately left out here to keep this notebook
  runnable end-to-end on free-tier Colab by anyone, including a grader without a paid
  subscription.
- **Corpus coverage gap, disclosed**: FOMC minutes before ~2010 are published under a
  legacy URL namespace this corpus's builder does not crawl (see
  `fomc_training_corpus_manifest.json`'s `known_coverage_gap` field); investigated as a
  genuine extension, found to require materially more engineering than its expected
  benefit justified at this stage — a scoped-out decision, not an oversight.
- **Regime-change boundary**: the corpus is deliberately frozen at Chair Powell's last
  meeting for stylistic homogeneity; extending past the transition would require either
  accepting a two-era, non-stationary corpus or training a separate post-transition
  model — left as future work.
- **Neither model is adopted as a single "final" choice over the other.** The
  character-level model is the required, fully self-contained deliverable; BPE is a
  clearly separated follow-up experiment that outperformed it here on bits-per-character,
  reversing two earlier rounds' findings once regularization was corrected. Both are kept
  and reported honestly rather than presenting only the model that currently looks best.
- **Char-KL and bits-per-character are sensitive to seed and sample size**; the values
  reported here come from a single seeded run and two sample prompts, not an average
  across seeds — a natural next step for anyone extending this further.

## Total Runtime

In [ ]:
elapsed_total = time.time() - NOTEBOOK_START
mins, secs = divmod(elapsed_total, 60)
print(f"total notebook runtime: {int(mins)} min {secs:.0f} s")
